In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, BooleanType, DateType
from pyspark.sql import Row
import random
import datetime

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("score", DoubleType(), True),
    StructField("passed", BooleanType(), True),
    StructField("date", DateType(), True)
])

names = ["Alice", "Bob", "Charlie", "David", "Eve"]
start_date = datetime.date(2020, 1, 1)

data = [
    Row(
        id=i,
        name=random.choice(names),
        score=round(random.uniform(0, 100), 2),
        passed=random.choice([True, False]),
        date=start_date + datetime.timedelta(days=random.randint(0, 365))
    )
    for i in range(1, 101)
]



In [0]:
df = spark.createDataFrame(data, schema)
#display(df)

In [0]:
df.printSchema()

In [0]:
def passOrFail(df):
    df = df.withColumn('yes_passed', df['score'] >= 35)
    return df


In [0]:
df = passOrFail(df)
df.display()

1.Read CSV  file with imnferred schema

In [0]:
df=spark.read.format("csv").option("header", "true").option("inferSchema","true").load("/Volumes/workspace/default/myvolume/orders_data/streaming_orders.csv")

In [0]:
%sql
SELECT current_catalog(), current_schema()

2.Read a CSV using explicit struct type

In [0]:
from pyspark.sql.types import *
schema=StructType([
    StructField("order_id",IntegerType()),
    StructField("customer_name",StringType()),
    StructField("product",StringType()),
    StructField("amount",IntegerType()),
    StructField("event_time",TimestampType())
])

df=spark.read.format("csv").option("header","true").schema(schema).load("/Volumes/workspace/default/myvolume/orders_data/streaming_orders.csv")

In [0]:
df.printSchema()

3.Select only three columns from a DataFrame.

In [0]:
df.select("order_id","customer_name","product")

In [0]:
#To show the columns use display()
display(df.select("order_id","name","product"))

4.Rename multiple columns in a dataframe

In [0]:
df=df.withColumnRenamed("amount","order_amount").withColumnRenamed("customer_name","name")
df.printSchema()

Filter amount > 40000

In [0]:
df3= df.filter(df.order_amount > 400)
df3.show()

Filter using multiple columns

In [0]:
from pyspark.sql.functions import col
df1=df.filter((col("name").like("K%")) & (col("order_amount") > 40000)).show()
df2=df.filter((col("name").like("K%")) | (col("order_amount") > 4000)).show()

Add new column

In [0]:
from pyspark.sql.functions import lit
df=df.withColumn("order_status",lit("Pending"))

Cast a string column to integer.

In [0]:
from pyspark.sql.functions import lit, col
df=df.withColumn("number_of_items",lit("1"))
df.withColumn("number_of_items",col("number_of_items").cast("int"))

Drop Duplicates

In [0]:
df=df.dropDuplicates()
display(df)

Replace null values with fillna

In [0]:
df=df.fillna({
    "order_id" : 0,
    "order_amount" : 0
})



Drop rows containing nulls.

In [0]:
df=df.dropna()

Count null values in every column.

In [0]:
from pyspark.sql.functions import col,when,sum
df.select([
    sum(when(col(c).isNull(),1).otherwise(0)).alias(c)
    for c in df.columns
]).show()

Save the DataFrame as Parquet and Delta.

In [0]:
df.write.format("delta").mode("append").save("/Volumes/workspace/default/myvolume/orders_data/order_delta")

In [0]:
df.write.mode("append").parquet("/Volumes/workspace/default/myvolume/orders_data/order_parquet")

Sort orders by amount descending

In [0]:
df.orderBy(col("amount").desc()).show()

Find the top 3 highest  orders by amount

In [0]:
df.sort(col("amount").desc()).limit(3).show()

Group by product and ahow average amount

In [0]:
from pyspark.sql.functions import avg
df.groupBy(col("product")).agg(avg("amount")).show()

Count no. of students that have passed or failed . (Use cell2 df)

In [0]:
count_passed=df.filter(col("passed") == True).count()
count_failed=df.filter(col("passed") == False).count()
print(f"Passed: {count_passed}, Failed: {count_failed}")

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType, BooleanType, DateType, TimestampType
from datetime import datetime, timedelta
import random
import builtins

customers = spark.read.table("samples.bakehouse.sales_customers")

schema = StructType([
    StructField("customerID", IntegerType(), True),
    StructField("amount", DoubleType(), True),
    StructField("order_date", DateType(), True),
    StructField("order_time", TimestampType(), True),
    StructField("product", StringType(), True),
    StructField("is_priority", BooleanType(), True),
    StructField("notes", StringType(), True)
])

products = ["Bread", "Cake", "Pie", "Cookie", "Muffin"]
notes = ["", "Urgent", "Gift", "Repeat", "First order", None]

base_date = datetime(2024, 6, 1)
data = []
for i in range(1, 101):
    amount = builtins.round(random.uniform(10, 500), 2)
    order_date = base_date + timedelta(days=random.randint(0, 30))
    order_time = datetime.combine(order_date, datetime.min.time()) + timedelta(hours=random.randint(0,23), minutes=random.randint(0,59))
    product = random.choice(products)
    is_priority = random.choice([True, False])
    note = random.choice(notes)
    data.append((i, amount, order_date.date(), order_time, product, is_priority, note))

orders = spark.createDataFrame(data, schema)

Join customers and orders using inner join to display:

customerID
first_name
last_name
product
amount

In [0]:

from pyspark.sql.functions import col

customers.alias("c").join(orders.alias("o"), col("c.customerID") == col("o.customerID"),"inner").select(col("c.customerID"),col("c.first_name"),col("c.last_name"),col("o.amount"),col("o.product")).show()

In [0]:
display(customers)

In [0]:
display(orders)

In [0]:
from pyspark.sql.functions import lit

# Select a few customerIDs from customers
sample_customers = customers.select("customerID").limit(5).collect()
sample_ids = [row.customerID for row in sample_customers]

# Create matching rows for orders
matching_rows = []
for cid in sample_ids:
    matching_rows.append((
        cid,
        builtins.round(random.uniform(20, 300), 2),
        datetime(2024, 6, 15).date(),
        datetime(2024, 6, 15, 10, 0),
        random.choice(products),
        True,
        "Practice join"
    ))

matching_orders = spark.createDataFrame(matching_rows, schema)
orders = orders.unionByName(matching_orders)

display(orders)

Display all customers, even those who haven't placed any order.

In [0]:
customers.alias("c").join(orders.alias("o"), col("c.customerID") == col("o.customerID"), "left").select(col("c.customerID"), col("c.first_name"), col("c.last_name"), col("o.amount"), col("o.product")).show()

Display all orders, even if a customer record is missing.

In [0]:
customers.join(orders,"customerID","right").select("customerID","product","amount","first_name").show(100)

Display every customer and every order, whether they match or not.

In [0]:
customers.join(orders,"customerID","full_outer").select("*").show()

Return only customers who have placed at least one order.

In [0]:
customers.join(orders,"customerID","left_semi").show()

Return customers who have never placed an order.


In [0]:
customers.join(orders,"customerID","left_anti").select("customerID","city","first_name").show()

Return customers who have never placed an order.


In [0]:
customers.join(orders,"customerID","left_anti").select("customerID","first_name","city").show()

Find customers whose order amount is greater than 5000 and from "India"

In [0]:
from pyspark.sql.functions import col
customers.join(orders,"customerID","inner").select("first_name").filter((col("amount")> 50) & (col("country") == "Japan")).show()

Find the total amount spent by each customer.


In [0]:

from pyspark.sql.functions import sum
customers.join(orders,"customerID","inner").groupBy("customerID","first_name").agg(sum("amount").alias("total_amount")).show()

Find the customer who spent the maximum amount overall.

In [0]:
from pyspark.sql.functions import max
customers.join(orders,"customerID","inner").orderBy(col("amount").desc()).select("first_name","customerID").limit(1).show()

Find the number of orders placed by each customer.


In [0]:
from pyspark.sql.functions import count
customers.join(orders.alias("o"),"customerID","inner").groupBy("customerID","first_name").agg(count("*").alias("order_count")).show()

Find the average order amount for every city.

In [0]:
from pyspark.sql.functions import avg
customers.join(orders,"customerID","inner").groupBy("city").agg(avg("amount").alias("avg_amount_per_city")).show()

Find the total sales for every country.

In [0]:
customers.join(orders,"customerID","inner").groupBy("country").agg(sum("amount").alias("total_amount")).show()

Find customers who placed more than 5 orders.


In [0]:
customers.join(orders,"customerID","inner").groupBy("customerID").agg(count("*").alias("order_count")).filter(col("order_count")>5).show()

Find the top 10 customers by total purchase amount.

In [0]:
customers.join(orders,"customerID","inner").groupBy("customerID").agg(sum("amount").alias("total_purchase_amount")).orderBy(col("total_purchase_amount").desc()).limit(10).show()

Find customers who ordered more than one unique product.



In [0]:
from pyspark.sql.functions import col, countDistinct
customers.join(orders,"customerID","inner").groupBy("customerID","first_name").agg(countDistinct("product").alias("unique_product")).filter(col("unique_product") >1).show()

In [0]:
from pyspark.sql.functions import col,current_date
customers.join(orders,"customerID","inner").filter(col("order_date") == current_date()).show()

Find customers who have never placed a priority order.


In [0]:
from pyspark.sql.functions import *
customers.join(orders,"customerID","inner").groupBy("customerID","first_name").agg(count(when(col("is_priority") == True,1)).alias("priority_count")).filter("priority_count = 0").show()

For every customer, show their latest order.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
window = Window.partitionBy("customerID").orderBy(col("order_date").desc())
customers.join(orders,"customerID","inner").withColumn("rn",row_number().over(window)).filter(col("rn") == 1).drop("rn").show()


For every customer, show their highest-value order.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
window = Window.partitionBy("customerID").orderBy(col("amount").desc())
customers.join(orders,"customerID","inner").withColumn("rn",row_number().over(window)).filter(col("rn") == 1).drop("rn").show()


Find the first customer who purchased every product.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *
window=Window.partitionBy("product").orderBy(col("order_date"),col("order_time"))
customers.join(orders,"customerID","inner").withColumn("rn",row_number().over(window)).filter(col("rn") == 1).drop("rn").show()


Find the most recent order in every city.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *
window=Window.partitionBy("city").orderBy(col("order_date").desc(),col("order_time").desc())
customers.join(orders,"customerID","inner").withColumn("rn",row_number().over(window)).filter(col("rn") == 1).drop("rn").show()

Rank customers by total purchase amount.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *
total_df = customers.join(orders,"customerID","inner").groupBy("customerID","first_name").agg(sum("amount").alias("total_amount"))
window = Window.orderBy(col("total_amount").desc())
total_df.withColumn("rank",rank().over(window)).show()

Find duplicate orders (same customer, product, amount, and order date).


In [0]:
orders.groupBy("customerID","amount","product","order_date").count().filter(col("count") > 1).show()

Remove duplicate orders while keeping the latest order_time.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *
window=Window.partitionBy("customerID","amount","product","order_date").orderBy(col("order_time").desc())
orders.withColumn("rn",row_number().over(window)).filter(col("rn") == 1).drop("rn").show()

> Find customers who haven't ordered in the last 30 days.

In [0]:
from pyspark.sql.functions import *
recent_orders = orders.filter(col("order_date") > date_sub(current_date(),30)).select(col("customerID")).distinct()
customers.join(recent_orders,"customerID","left_anti").show()